# Project 1 - Part 2

In [0]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.appName("my_project_1").getOrCreate()


Importing all spark data types and spark functions for your convenience.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

The type of household ID was originally a string, but we changed it to an integer to match its type in the reference data. This was done to enable a successful join.

In [0]:
# Read a CSV into a dataframe
# There is a smarter version, that will first check if there is a Parquet file and use it
def load_csv_file(filename, schema):
  # Reads the relevant file from distributed file system using the given schema

  allowed_files = {'Daily program data': ('Daily program data', "|"),
                   'demographic': ('demographic', "|")}

  if filename not in allowed_files.keys():
    print(f'You were trying to access unknown file \"{filename}\". Only valid options are {allowed_files.keys()}')
    return None

  filepath = allowed_files[filename][0]
  dataPath = f"dbfs:/mnt/coursedata2024/fwm-stb-data/{filepath}"
  delimiter = allowed_files[filename][1]

  df = spark.read.format("csv")\
    .option("header","false")\
    .option("delimiter",delimiter)\
    .schema(schema)\
    .load(dataPath)
  return df

# This dict holds the correct schemata for easily loading the CSVs
schemas_dict = {'Daily program data':
                  StructType([
                    StructField('prog_code', StringType()),
                    StructField('title', StringType()),
                    StructField('genre', StringType()),
                    StructField('air_date', StringType()),
                    StructField('air_time', StringType()),
                    StructField('Duration', FloatType())
                  ]),
                'viewing':
                  StructType([
                    StructField('device_id', StringType()),
                    StructField('event_date', StringType()),
                    StructField('event_time', IntegerType()),
                    StructField('mso_code', StringType()),
                    StructField('prog_code', StringType()),
                    StructField('station_num', StringType())
                  ]),
                'viewing_full':
                  StructType([
                    StructField('mso_code', StringType()),
                    StructField('device_id', StringType()),
                    StructField('event_date', IntegerType()),
                    StructField('event_time', IntegerType()),
                    StructField('station_num', StringType()),
                    StructField('prog_code', StringType())
                  ]),
                'demographic':
                  StructType([StructField('household_id',IntegerType()),
                    StructField('household_size',IntegerType()),
                    StructField('num_adults',IntegerType()),
                    StructField('num_generations',IntegerType()),
                    StructField('adult_range',StringType()),
                    StructField('marital_status',StringType()),
                    StructField('race_code',StringType()),
                    StructField('presence_children',StringType()),
                    StructField('num_children',IntegerType()),
                    StructField('age_children',StringType()), #format like range - 'bitwise'
                    StructField('age_range_children',StringType()),
                    StructField('dwelling_type',StringType()),
                    StructField('home_owner_status',StringType()),
                    StructField('length_residence',IntegerType()),
                    StructField('home_market_value',StringType()),
                    StructField('num_vehicles',IntegerType()),
                    StructField('vehicle_make',StringType()),
                    StructField('vehicle_model',StringType()),
                    StructField('vehicle_year',IntegerType()),
                    StructField('net_worth',IntegerType()),
                    StructField('income',StringType()),
                    StructField('gender_individual',StringType()),
                    StructField('age_individual',IntegerType()),
                    StructField('education_highest',StringType()),
                    StructField('occupation_highest',StringType()),
                    StructField('education_1',StringType()),
                    StructField('occupation_1',StringType()),
                    StructField('age_2',IntegerType()),
                    StructField('education_2',StringType()),
                    StructField('occupation_2',StringType()),
                    StructField('age_3',IntegerType()),
                    StructField('education_3',StringType()),
                    StructField('occupation_3',StringType()),
                    StructField('age_4',IntegerType()),
                    StructField('education_4',StringType()),
                    StructField('occupation_4',StringType()),
                    StructField('age_5',IntegerType()),
                    StructField('education_5',StringType()),
                    StructField('occupation_5',StringType()),
                    StructField('polit_party_regist',StringType()),
                    StructField('polit_party_input',StringType()),
                    StructField('household_clusters',StringType()),
                    StructField('insurance_groups',StringType()),
                    StructField('financial_groups',StringType()),
                    StructField('green_living',StringType())
                  ])
}

# Read demogrphic data


In [0]:
%%time
# demographic data filename is 'demographic'
demo_df = load_csv_file('demographic', schemas_dict['demographic'])
demo_df.count()
demo_df.printSchema()
print(f'demo_df contains {demo_df.count()} records!')
display(demo_df.limit(6))

root
 |-- household_id: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- num_adults: integer (nullable = true)
 |-- num_generations: integer (nullable = true)
 |-- adult_range: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- race_code: string (nullable = true)
 |-- presence_children: string (nullable = true)
 |-- num_children: integer (nullable = true)
 |-- age_children: string (nullable = true)
 |-- age_range_children: string (nullable = true)
 |-- dwelling_type: string (nullable = true)
 |-- home_owner_status: string (nullable = true)
 |-- length_residence: integer (nullable = true)
 |-- home_market_value: string (nullable = true)
 |-- num_vehicles: integer (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_model: string (nullable = true)
 |-- vehicle_year: integer (nullable = true)
 |-- net_worth: integer (nullable = true)
 |-- income: string (nullable = true)
 |-- gender_individual: string (nullable = 

household_id,household_size,num_adults,num_generations,adult_range,marital_status,race_code,presence_children,num_children,age_children,age_range_children,dwelling_type,home_owner_status,length_residence,home_market_value,num_vehicles,vehicle_make,vehicle_model,vehicle_year,net_worth,income,gender_individual,age_individual,education_highest,occupation_highest,education_1,occupation_1,age_2,education_2,occupation_2,age_3,education_3,occupation_3,age_4,education_4,occupation_4,age_5,education_5,occupation_5,polit_party_regist,polit_party_input,household_clusters,insurance_groups,financial_groups,green_living
15,2,2,1,000000000000100000000,S,B,null,null,0000000000000000000,000000000000000,S,O,5,E,null,null,null,null,6,4,M,60,4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,D,443,02C3,08C3,null
24,2,2,1,000000000100000000000,null,W,null,null,0000000000000000000,000000000000000,M,O,null,F,null,null,null,null,7,7,F,46,3,Z,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,R,223,09O3,03O3,null
26,null,null,null,000000000000000000000,null,null,null,null,0000000000000000000,000000000000000,S,null,null,F,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,46G,04CG,08CG,null
28,3,2,2,000000110000000000000,S,W,Y,1,0000010000000000000,000001000000000,S,O,3,H,null,null,null,null,5,7,M,38,2,4,null,null,34,1,7,null,null,null,null,null,null,null,null,null,null,V,473,11R3,09C3,1
35,1,1,1,000000000100000000000,null,W,null,null,0000000000000000000,000000000000000,null,null,null,G,null,null,null,null,4,null,M,50,2,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,D,523,13C3,08C3,null
36,null,null,null,000000000000000000000,null,null,null,null,0000000000000000000,000000000000000,null,null,null,G,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,51G,10RG,10RG,null


CPU times: user 35.8 ms, sys: 10.2 ms, total: 46 ms
Wall time: 25.8 s


# Read Daily program data

In [0]:
%%time
# daily_program data filename is 'Daily program data'
daily_prog_df = load_csv_file('Daily program data', schemas_dict['Daily program data'])

daily_prog_df.printSchema()
print(f'daily_prog_df contains {daily_prog_df.count()} records!')
display(daily_prog_df.limit(6))

root
 |-- prog_code: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- air_date: string (nullable = true)
 |-- air_time: string (nullable = true)
 |-- Duration: float (nullable = true)

daily_prog_df contains 13194849 records!


prog_code,title,genre,air_date,air_time,Duration
EP000000250035,21 Jump Street,Crime drama,20151219,050000,60.0
EP000000250035,21 Jump Street,Crime drama,20151219,110000,60.0
EP000000250063,21 Jump Street,Crime drama,20151219,180000,60.0
EP000000510007,A Different World,Sitcom,20151219,100000,30.0
EP000000510008,A Different World,Sitcom,20151219,103000,30.0
EP000000510159,A Different World,Sitcom,20151219,080300,29.0


CPU times: user 18.6 ms, sys: 1.85 ms, total: 20.5 ms
Wall time: 9.85 s


# Read viewing data

In [0]:
dataPath = "dbfs:/FileStore/ddm/10m_viewing"

viewing10m_df = spark.read.format("csv")\
    .option("header","true")\
    .option("delimiter",",")\
    .schema(schemas_dict['viewing_full'])\
    .load(dataPath)

display(viewing10m_df.limit(6))
print(f'viewing10m_df contains {viewing10m_df.count()} rows!')

mso_code,device_id,event_date,event_time,station_num,prog_code
01540,0000000050f3,20150222,193802,61812,EP009279780033
01540,0000000050f3,20150222,195314,31709,EP021056430002
01540,0000000050f3,20150222,200151,61812,EP009279780033
01540,000000005518,20150222,111139,46784,EP004891370013
01540,000000005518,20150222,190000,14771,EP012124070127
01540,000000005518,20150222,200000,14771,EP010237320166


viewing10m_df contains 9935852 rows!


# Read reference data

Note that we removed the 'System Type' column.

In [0]:
# Read the new parquet
ref_data_schema = StructType([
    StructField('device_id', StringType()),
    StructField('dma', StringType()),
    StructField('dma_code', StringType()),
    StructField('household_id', IntegerType()),
    StructField('zipcode', IntegerType())
])

# Reading as a Parquet
dataPath = f"dbfs:/ddm_course_staff_files/ref_data"
ref_data = spark.read.format('parquet') \
                    .option("inferSchema","true")\
                    .load(dataPath)
                    
display(ref_data.limit(6))
print(f'ref_data contains {ref_data.count()} rows!')

device_id,dma,dma_code,household_id,zipcode
0000000050f3,Toledo,547,1471346,43609
000000006785,Amarillo,634,1924512,79119
000000007320,Lake Charles,643,3154808,70634
000000007df9,Lake Charles,643,1924566,70601
000000009595,Lexington,541,1600886,40601
000000009c6a,Houston,618,1924713,77339


ref_data contains 704172 rows!


## Q2.1

#### Query 1 - Top 5 Generes
אנו מפרשים את הדרישה כמספר האנשים הייחודיים שצפו בז'אנר מסוים

In [0]:

# Filter relevant columns and remove duplicates from each DataFrame
ref = ref_data.select("device_id", "household_id").distinct()
daily_prog = daily_prog_df.select("prog_code", "genre").distinct()
viewing10m = viewing10m_df.select("device_id", "prog_code").distinct()
demo = demo_df.select("household_id", "household_size").distinct()

# Convert the genre string into an array to allow checking for specific genres
daily_prog = daily_prog.withColumn("genres_arr", split(col("genre"), ",")) \
                       .drop("genre")

# Join all data sources
all_data = demo.join(ref, on="household_id", how="inner") \
               .join(viewing10m, on="device_id", how="inner") \
               .join(daily_prog, on="prog_code", how="inner")

# Explode the array of genres so that each genre gets its own row
all_data = all_data.withColumn("genre", explode("genres_arr"))

# Keep only one record per (household_id, genre) pair to avoid double-counting
unique_households_per_genre = all_data.select("household_id", "genre", "household_size").dropDuplicates(["household_id", "genre"])

# Group by genre to get the total number of viewers (household sizes summed)
genre_viewers = unique_households_per_genre.groupBy("genre") \
                                           .agg(sum("household_size").alias("total_viewers_per_genre")) \
                                           .orderBy(col("total_viewers_per_genre").desc())

# Display top 5 genres and their total viewers
top5_genres = genre_viewers.limit(5)
display(top5_genres)


genre,total_viewers_per_genre
News,615305
Reality,610476
Talk,537723
Comedy,509672
Sitcom,502753


#### Query 2 - Top 5 DMA

In [0]:
# Select relevant columns and remove duplicates
ref = ref_data.select("device_id", "DMA", "household_id").distinct()
demo = demo_df.select("household_id", "household_size").distinct()

# Count number of unique devices per DMA
devices_per_dma = ref.groupBy("DMA").agg(countDistinct("device_id").alias("count_devices_per_DMA")) ###########

# Get the top 5 DMAs with the highest number of devices
top5_dma = devices_per_dma.orderBy(col("count_devices_per_DMA").desc()).limit(5)

# Remove potential duplicates of (DMA, household_id) before joining
ref = ref.select("DMA", "household_id").distinct()

# Join top 5 DMAs back to full ref
top5_ref = ref.join(top5_dma, on="DMA", how="inner")

# Join the top DMA household data with demographic data to get household sizes
joined = top5_ref.join(demo, on="household_id", how="inner")

# Group by DMA and sum household sizes
result = joined.groupBy("DMA").agg(sum("household_size").alias("total_people_per_DMA"))

# Display the DMAs ordered by total population size (descending)
display(result.orderBy(col("total_people_per_DMA").desc()))

DMA,total_people_per_DMA
Charleston-Huntington,60656
Wilkes Barre-Scranton-Hztn,42844
Seattle-Tacoma,35124
Little Rock-Pine Bluff,31652
Toledo,24108


#### Query 3 - Top 5 Programs
בדומה לשאילתה 1 - אנו מפרשים את הדרישה כמספר האנשים הייחודיים שצפו

In [0]:
# Query 3

# Step 1: Filter relevant columns and drop duplicates
ref = ref_data.select("device_id", "household_id").distinct()
daily_prog = daily_prog_df.select("prog_code", "title").distinct()
viewing10m = viewing10m_df.select("device_id", "prog_code").distinct()
demo = demo_df.select("household_id", "household_size", "presence_children").distinct()

# Step 2: Filter to households with children
households_with_kids = demo.filter(col("presence_children") == "Y")

# Step 3: Join only for household with kids to find top 5 titles
data_with_kids = households_with_kids.join(ref, "household_id", "inner") \
                                     .join(viewing10m, "device_id", "inner") \
                                     .join(daily_prog, "prog_code", "inner")

# Step 4: Filter and aggregate to get top 5 programs (based on households with kids)
top_titles = data_with_kids.filter(col("title").isNotNull()) \
                           .select("title", "household_id") \
                           .distinct() \
                           .groupBy("title") \
                           .agg(count("*").alias("num_kid_households")) \
                           .orderBy(col("num_kid_households").desc()) \
                           .limit(5)

# Step 5: Get total number of people (from all households) who viewed those 5 titles
# Join again with full household info
full_viewers_data = demo.join(ref, "household_id", "inner") \
                        .join(viewing10m, "device_id", "inner") \
                        .join(daily_prog, "prog_code", "inner")

# Restrict to just the selected titles
top5_titles_list = [row["title"] for row in top_titles.collect()]
final = full_viewers_data.filter(col("title").isin(top5_titles_list)) \
                         .select("title", "household_id", "household_size") \
                         .distinct() \
                         .groupBy("title") \
                         .agg(sum("household_size").alias("total_viewers_all_households")) \
                         .orderBy(col("total_viewers_all_households").desc())

display(final)


title,total_viewers_all_households
College Basketball,155682
Paid Programming,147968
SportsCenter,111622
The Big Bang Theory,104211
Today,92618


## Q2.2

In [0]:
# Filter relevant columns and remove duplicates from each DataFrame
ref_filtered  = ref_data.select("DMA", "household_id", "device_id").distinct()
daily_prog_filtered  = daily_prog_df.select("prog_code", "genre").distinct()
viewing_filtered = viewing10m_df.select("device_id", "prog_code").distinct()
demo_filtered  = demo_df.select("household_id", "net_worth", "income").distinct()

# Calculate WEALTH SCORE

# Convert income categories and numeric strings into numeric values
# - Income values 'A'-'D' are mapped to 10-13
# - Income values '0'-'9' are cast as integers
demo_converted = demo_filtered.withColumn(
    "income_numeric",
    when(col("income") == "A", lit(10))
    .when(col("income") == "B", lit(11))
    .when(col("income") == "C", lit(12))
    .when(col("income") == "D", lit(13))
    .otherwise(col("income").cast("int"))
)

# Remove device-level duplicates by keeping one row per household per DMA
household_dma = ref_filtered.select("household_id", "DMA").distinct()

# Join household DMA info with demographic data (income & net worth)
demo_ref = demo_converted.join(household_dma, on="household_id", how="inner")

# Compute average net worth and income per DMA
dma_avgs = demo_ref.groupBy("DMA").agg(
    avg("net_worth").alias("avg_net_worth"),
    avg("income_numeric").alias("avg_income")
)

# Compute global maximum values of net worth and income
max_vals = demo_ref.agg(
    max("net_worth").alias("max_net_worth"),
    max("income_numeric").alias("max_income")
).collect()[0]

# Compute normalized wealth score for each DMA:
# wealth_score = (avg_net_worth / global_max_net_worth) + (avg_income / global_max_income)
dma_wealth = dma_avgs.withColumn(
    "wealth_score",
    (col("avg_net_worth") / max_vals["max_net_worth"]) +
    (col("avg_income") / max_vals["max_income"])
).orderBy(col("wealth_score").desc())

# Select top 10 DMAs by wealth score
top_DMA = dma_wealth.limit(10)
top_dma_list = [row["DMA"] for row in top_DMA.select("DMA").collect()]

# Distinct most popular genres per DMA

# Explode multiple genres per program into individual rows
genres_df = daily_prog_filtered.withColumn(
    "genre", explode(split(col("genre"), ","))
).withColumn("genre", trim(col("genre"))) \
 .select("prog_code", "genre") \
 .distinct()

# Calculate genre popularity per DMA
# Popularity is based on the count of distinct devices watching programs of that genre
genre_popularity = viewing_filtered \
    .join(ref_filtered, "device_id", "inner") \
    .join(genres_df, "prog_code", "inner") \
    .groupBy("DMA", "genre") \
    .agg(countDistinct("device_id").alias("popularity")) \
    .withColumnRenamed("DMA", "DMA_UPPER")

# Set to keep track of globally selected genres (to avoid duplicates)
selected_genres = set()

# List to accumulate final output for each DMA
dma_genre_selections = []

# Iterate over top 10 DMAs by wealth score
for row in top_DMA.collect():
    dma = row["DMA"]
    wealth = row["wealth_score"]
    
    # Filter top genres for this DMA excluding genres already selected
    available_genres = genre_popularity \
        .filter((col("DMA_UPPER") == dma) & (~col("genre").isin(list(selected_genres)))) \
        .orderBy(col("popularity").desc()) \
        .limit(11) \
        .select("genre") \
        .rdd.flatMap(lambda x: x) \
        .collect()
    
    # Update selected genres to avoid duplication across DMAs
    selected_genres.update(available_genres)
    
    # Save results for current DMA
    dma_genre_selections.append((dma, wealth, available_genres))

# Convert final list into a DataFrame with three columns:
# 1.DMA name
# 2.Computed wealth score
# 3.Selected genres (list)
result_df = spark.createDataFrame(dma_genre_selections, ["DMA", "WEALTH SCORE", "GENRES"])

# Display the final result
display(result_df)


DMA,WEALTH SCORE,GENRES
San Antonio,1.623931623931624,List()
San Francisco-Oak-San Jose,1.5138661139259941,"List(Reality, News, Comedy, Music, Sitcom, Talk, Drama, Documentary, Adventure, Children, Action)"
Baltimore,1.4813042534625838,List()
Sacramnto-Stkton-Modesto,1.4164879527121452,"List(Entertainment, Crime drama, Consumer, Animated, Newsmagazine, Suspense, Fantasy, Crime, Special, Mystery, Sports event)"
"Bend, OR",1.3998651038874286,"List(Shopping, Sports non-event, Game show, House/garden, Educational, Law, Travel, Public affairs, Interview, Cooking, How-to)"
Austin,1.389955092480879,"List(Home improvement, Science fiction, Politics, Basketball, Romance, Sports talk, History, Bus./financial, Horror, Medical, Science)"
Seattle-Tacoma,1.374988109195418,"List(Outdoors, Animals, Western, Nature, Comedy-drama, Religious, Romance-comedy, Paranormal, Health, Soap, Historical drama)"
Houston,1.3579162058465934,"List(Weather, Golf, Fashion, Auto, Awards, Biography, Auto racing, War, Football, Docudrama, Musical)"
Detroit,1.3305256247490695,"List(Hockey, Variety, Fishing, Collectibles, Technology, Hunting, Baseball, Parenting, Auction, Anthology, Action sports)"
Harrisburg-Lncstr-Leb-York,1.2969389678512706,"List(Community, Soccer, Pro wrestling, Mixed martial arts, Self improvement, Art, Dog show, Aviation, Military, Dance, Figure skating)"
